# Kaggle Direct Local LLM Runner

This notebook runs the ARC agent with a local Hugging Face model loaded directly through `transformers`.
Use this when `vllm` is not installed or you want a simpler fully-offline Kaggle path.

In [ ]:
from pathlib import Path

def detect_repo_dir() -> Path:
    candidates = [Path.cwd(), Path('/kaggle/working/ARC-AGI-3-Agents'), Path('/kaggle/working')]
    for base in candidates:
        if (base / 'main.py').exists() and (base / 'agents').exists():
            return base
    for child in Path('/kaggle/working').glob('*'):
        if child.is_dir() and (child / 'main.py').exists() and (child / 'agents').exists():
            return child
    raise FileNotFoundError('Could not auto-detect repo dir under /kaggle/working')

REPO_DIR = detect_repo_dir()
MODEL_PATH = Path('/kaggle/input/your-model-dir')
GAME_ID = 'locksmith'
AGENT_NAME = 'directlocalllm'
MAX_FRAME_CHARS = 2500
DIRECT_DTYPE = 'auto'
DIRECT_DEVICE_MAP = 'auto'
DIRECT_LOAD_IN_4BIT = False
DIRECT_TRUST_REMOTE_CODE = False
DIRECT_MAX_NEW_TOKENS = 160
DIRECT_TEMPERATURE = 0.0

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert MODEL_PATH.exists(), f'Model path not found: {MODEL_PATH}'
print('repo:', REPO_DIR)
print('model:', MODEL_PATH)

In [ ]:
import os
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

## Optional dependency checks

In [ ]:
import shutil
import subprocess

for tool in ['python', 'uv']:
    print(tool, '->', shutil.which(tool))

for mod in ['torch', 'transformers', 'accelerate']:
    try:
        __import__(mod)
        print(mod, 'OK')
    except Exception as exc:
        print(mod, 'ERR', exc)

## Write `.env` for offline direct execution

In [ ]:
from textwrap import dedent

env_text = dedent(f'''
DEBUG=False
RECORDINGS_DIR=recordings
SCHEME=http
HOST=127.0.0.1
PORT=8000
ARC_BASE_URL=https://three.arcprize.org/
OPERATION_MODE=local
ENVIRONMENTS_DIR=environment_files
LLM_MAX_FRAME_CHARS={MAX_FRAME_CHARS}
AGENTOPS_API_KEY=
ARC_API_KEY=
DIRECT_LLM_MODEL_PATH={MODEL_PATH}
DIRECT_LLM_DEVICE_MAP={DIRECT_DEVICE_MAP}
DIRECT_LLM_DTYPE={DIRECT_DTYPE}
DIRECT_LLM_LOAD_IN_4BIT={str(DIRECT_LOAD_IN_4BIT)}
DIRECT_LLM_TRUST_REMOTE_CODE={str(DIRECT_TRUST_REMOTE_CODE)}
DIRECT_LLM_MAX_NEW_TOKENS={DIRECT_MAX_NEW_TOKENS}
DIRECT_LLM_TEMPERATURE={DIRECT_TEMPERATURE}
''').strip() + '\n'

Path('.env').write_text(env_text)
print(Path('.env').read_text())

## Optional model folder sanity check

In [ ]:
from pathlib import Path

for path in sorted(MODEL_PATH.glob('*'))[:20]:
    print(path.name)

## Run the direct local agent

In [ ]:
import subprocess

run_cmd = ['uv', 'run', 'main.py', '--agent', AGENT_NAME, '--game', GAME_ID]
print('running:', ' '.join(run_cmd))
result = subprocess.run(run_cmd, text=True, capture_output=True)
print('returncode:', result.returncode)
print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)

## Inspect logs

In [ ]:
from pathlib import Path

path = Path('logs.log')
print(f'===== {path} =====')
if path.exists():
    text = path.read_text(errors='ignore')
    print(text[-12000:])
else:
    print('missing')